<a id="top"></a>

# Demo: schemas as a teaching tool

```yaml
title: "Demo: schemas as a teaching tool"
keywords: parameter schema, loose vs tight schema, validation error, informative vs opaque, shape not truth, book_meeting, recoverable error, tool_calls, ToolMessage, bind_tools, teacher demo
estimated duration: 14
```

> **Lesson:** L08. Teacher demo — Demo 3 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L08/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` before running. Clear outputs before committing.
>
> **The client is LangChain's `ChatAnthropic`** (from L03/L07). `bind_tools([fn])` registers a plain typed function as a tool — its definition inferred from the function's name, docstring, and type hints; the model's choice shows up in `AIMessage.tool_calls`, and a tool result (including an error) goes back as a `ToolMessage`. The API key still loads through the config seam (`require_anthropic_key`); never hard-coded.
>
> The point to land: **a tight schema converts ambiguity into validation errors the model can recover from; a loose schema pushes the ambiguity into a silent runtime guess.** And the error *shape* decides whether the model recovers — an error is a prompt for its next turn.

## Contents

- [1. Setup — loose vs tight book_meeting, and two error styles](#1-setup--loose-vs-tight-book_meeting-and-two-error-styles)
- [2. The loose schema — silent wrongness](#2-the-loose-schema--silent-wrongness)
- [3. Tight schema + opaque errors — a noisy, unhelpful loop](#3-tight-schema--opaque-errors--a-noisy-unhelpful-loop)
- [4. Tight schema + informative errors — failing productively](#4-tight-schema--informative-errors--failing-productively)
- [5. Schemas validate shape, not truth](#5-schemas-validate-shape-not-truth)
- [6. Takeaways](#6-takeaways)

## 1. Setup — loose vs tight book_meeting, and two error styles

One conceptual tool, `book_meeting`, in two schema shapes: **loose** (one free-form `details` string) and **tight** (typed, all-required fields with per-parameter constraints). Each is a plain typed function; `bind_tools` infers the schema from its signature and docstring. The tight tool's validator comes in two styles — **informative** errors that name the field + constraint, and **opaque** errors that don't. One ambiguous prompt drives all three runs. Anchor model: **Claude Sonnet 4.6**.

In [ ]:
from typing import Annotated

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage

from fluffy_potato_curriculum.common.config import require_anthropic_key

SONNET = "claude-sonnet-4-6"


# The LOOSE tool: one free-form field. bind_tools infers the schema from the signature
# and docstring -- so a plain typed function IS the tool.
def book_meeting_loose(
    details: Annotated[str, "the meeting details"],
) -> str:
    """Book a meeting."""
    return details


# The TIGHT tool: typed, all-required fields, each with a per-parameter constraint. Same
# name, but the schema pins down shape. Its docstring tells the model errors are recoverable.
def book_meeting_tight(
    attendee_email: Annotated[str, "RFC 5322 email, e.g. 'priya@example.com'."],
    start_iso: Annotated[str, "ISO 8601 start, e.g. '2024-11-05T14:00:00'."],
    duration_minutes: Annotated[int, "integer between 15 and 240."],
    title: Annotated[str, "short meeting title."],
) -> str:
    """Book a calendar meeting. All fields are required and validated; on a bad field you
    will receive a structured validation error naming the field to fix."""
    return f"booked {title} with {attendee_email}"


def validate_informative(args: dict[str, object]) -> dict[str, object]:
    """Validate tight-schema args; return a structured {error, field, message} or a booking."""
    email = str(args.get("attendee_email", ""))
    if "@" not in email:
        return {
            "error": "validation",
            "field": "attendee_email",
            "message": f"must be an email, got {email!r}",
        }
    minutes = args.get("duration_minutes")
    if not isinstance(minutes, int) or not (15 <= minutes <= 240):
        return {
            "error": "validation",
            "field": "duration_minutes",
            "message": f"must be an integer 15..240, got {minutes!r}",
        }
    return {"booked": True, "with": email, "minutes": minutes}


def validate_opaque(args: dict[str, object]) -> dict[str, object]:
    """Same checks, but every failure collapses to a useless generic message."""
    res = validate_informative(args)
    if "error" in res:
        return {"error": "bad input"}  # no field, no constraint
    return res


model = ChatAnthropic(model=SONNET, api_key=require_anthropic_key(), max_tokens=400)


def show_turn(resp: AIMessage) -> None:
    """Print a model turn: any text content, plus each tool call (name + args)."""
    print("tool_calls:", [c["name"] for c in resp.tool_calls])
    if resp.content:
        print("  text  ", repr(resp.content))
    for call in resp.tool_calls:
        print(f"  call   {call['name']}  args={call['args']}")


# Ambiguous on purpose: no email, vague "afternoon", relative date.
PROMPT = "Book a 90-minute design review with Priya next Tuesday afternoon."
print("setup ready (live model:", SONNET, ")")

[↑ Back to top](#top)

## 2. The loose schema — silent wrongness

Run the ambiguous prompt against the **loose** schema. The model packs everything into the `details` string and the tool 'succeeds'. Whether the meeting was booked *correctly* — with whose email, at what exact time — is **impossible to tell** from the conversation. This is the worst outcome: it looks like it worked.

In [ ]:
resp = await model.bind_tools([book_meeting_loose]).ainvoke([HumanMessage(PROMPT)])
show_turn(resp)
call = resp.tool_calls[0] if resp.tool_calls else None
if call is not None:
    print("\nthe tool got ONE blob:", call["args"])
    print("booked correctly? impossible to tell from here.")

[↑ Back to top](#top)

## 3. Tight schema + opaque errors — a noisy, unhelpful loop

Run the same prompt against the **tight** schema with **opaque** errors. The model guesses each field and submits; the validator returns `{'error': 'bad input'}`. That goes back as a `ToolMessage` with `status='error'`, and the model gets another shot — it typically retries with *another* guess, still opaque, still wrong. Two rounds below make the loop visible.

In [ ]:
tight_model = model.bind_tools([book_meeting_tight])
messages: list[BaseMessage] = [HumanMessage(PROMPT)]
for round_num in range(2):
    resp = await tight_model.ainvoke(messages)
    print(f"--- round {round_num + 1} ---")
    show_turn(resp)
    if not resp.tool_calls:
        break
    call = resp.tool_calls[0]
    result = validate_opaque(call["args"])
    print("  tool returned:", result)
    messages.append(resp)
    # An opaque error goes back as a ToolMessage with status="error" -- no field, no fix.
    messages.append(ToolMessage(content=str(result), tool_call_id=call["id"], status="error"))

[↑ Back to top](#top)

## 4. Tight schema + informative errors — failing productively

Same prompt, same tight schema, but now **informative** errors. The model submits, gets a structured error naming the offending field, and on the next turn either **asks the user** for the missing detail (Priya's email) or **fixes the field** and retries. The error taught the model what to do.

In [ ]:
messages = [HumanMessage(PROMPT)]
for round_num in range(3):
    resp = await tight_model.ainvoke(messages)
    print(f"--- round {round_num + 1} ---")
    show_turn(resp)
    if not resp.tool_calls:
        print("  (model stopped calling — asked the user or finished)")
        break
    call = resp.tool_calls[0]
    result = validate_informative(call["args"])
    print("  tool returned:", result)
    messages.append(resp)
    # Informative failures go back as status="error" (naming the field); a booking is a
    # normal successful ToolMessage.
    failed = "error" in result
    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=call["id"],
            status="error" if failed else "success",
        )
    )
    if "booked" in result:
        break

[↑ Back to top](#top)

## 5. Schemas validate shape, not truth

Here's a subtle trap worth naming: the model may guess `priya@example.com` so confidently that it **passes** email-format validation — even though no such user may exist. The tight schema catches *malformed* arguments; it does **not** catch *false* ones. That gap is exactly why **runtime errors** ([L0809](L0809_lecture.ipynb)) are the second line of defense.

In [ ]:
fake_but_well_formed = {
    "attendee_email": "priya@example.com",  # valid shape, possibly nonexistent person
    "start_iso": "2024-11-05T14:00:00",
    "duration_minutes": 90,
    "title": "Design review",
}
print("validation says:", validate_informative(fake_but_well_formed))
print("note: the shape is valid even if the address is fake — shape != truth.")

[↑ Back to top](#top)

## 6. Takeaways

- The **loose** schema *appears* to work and is the worst outcome (silent wrongness). The tight schema with **opaque** errors fails noisily but unhelpfully. The tight schema with **informative** errors fails *productively* — the model learns from each turn.
- A tight schema — required fields, narrow types, per-field constraints — turns the model's fuzziness ([L02](../L02/L0201_intro.md)) into **recoverable validation errors**.
- An **error message is a prompt** for the model's next turn — write it like a system message, not a Python traceback. The [L0808 lab](L0808_lab_empty.ipynb) drills exactly this rewrite, offline.
- Schemas validate **shape, not truth** — the bridge to [L0809](L0809_lecture.ipynb)'s runtime errors and side effects.

[↑ Back to top](#top)